## Pokémon TCG AI: Mega Lucario Rule-Based Agent

### 💡 What is a Rule-Based Agent?
Currently, the top of the leaderboard is dominated by **Rule-Based Heuristic Agents**. This means instead of using a deep neural network, the agent looks at all possible valid moves (e.g., Attack, Retreat, Play a Supporter) and assigns a numerical "score" to each move based on hardcoded logic. The move with the highest score is executed.
 
### 🎯 The Deck: Mega Lucario ex
This notebook implements a highly competitive Mega Lucario ex deck. It focuses on taking early knockouts and using Fighting-type synergies.
 
### 🛠️ What this notebook does:
1. **Defines the Agent**: We write our `main.py` agent file.
2. **Local Simulation**: We test our agent locally against a Random Agent to ensure it works.
3. **Submission Packaging**: We compress the agent into a `submission.tar.gz` ready to be submitted to the leaderboard.
 
If this helps you climb the leaderboard, an **upvote** is greatly appreciated!

---
## 1. Writing the Agent (`main.py`) ✍️
Kaggle requires our agent to be inside a file named `main.py`. We use the `%%writefile` magic command to save the contents of this cell directly to that file.
 
The agent function `agent(obs_dict)` takes in the current game state and returns the index of the action it wants to take.

In [1]:
!cp -r /kaggle/input/datasets/nursrijan/cg-lib-mod/cg /kaggle/working

In [2]:
%%writefile main.py
import os
from collections import defaultdict
from cg.api import AreaType, CardType, EnergyType, Observation, SelectContext, OptionType, Card, Pokemon, all_card_data, to_observation_class

# ---------------------------------------------------------
# 1. DECK DEFINITION (Mega Lucario ex Archetype)
# ---------------------------------------------------------
# A legal deck must have exactly 60 cards. 
# We hardcode the card IDs for our Fighting-type strategy.
my_deck = (
    [673]*2 +  # Makuhita
    [674]*2 +  # Hariyama
    [675]*2 +  # Lunatone
    [676]*3 +  # Solrock
    [677]*3 +  # Riolu
    [678]*4 +  # Mega Lucario ex
    [1102]*4 + # Dusk Ball
    [1123]*2 + # Switch
    [1141]*4 + # Premium Power Pro
    [1142]*4 + # Fighting Gong
    [1152]*4 + # Poke Pad
    [1159]*1 + # Hero Cape
    [1182]*2 + # Boss Orders
    [1192]*4 + # Carmine
    [1227]*4 + # Lillie's Determination
    [1252]*2 + # Gravity Mountain
    [6]*13     # Basic Fighting Energy
)

# Fetch card metadata database (HP, Attacks, Weaknesses, etc.)
all_card = all_card_data()
card_table = {c.cardId: c for c in all_card}

# ---------------------------------------------------------
# 2. HELPER FUNCTIONS
# ---------------------------------------------------------
def get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:
    """Safely extracts a Card or Pokemon object from specific zones (Hand, Bench, Active, etc.)."""
    ps = obs.current.players[player_index]
    match area:
        case AreaType.DECK: return obs.select.deck[index]
        case AreaType.HAND: return ps.hand[index]
        case AreaType.DISCARD: return ps.discard[index]
        case AreaType.ACTIVE: return ps.active[index]
        case AreaType.BENCH: return ps.bench[index]
        case AreaType.PRIZE: return ps.prize[index]
        case AreaType.STADIUM: return obs.current.stadium[index]
        case AreaType.LOOKING: return obs.current.looking[index]
        case _: return None

# ---------------------------------------------------------
# 3. MAIN AGENT FUNCTION
# ---------------------------------------------------------
def agent(obs_dict: dict) -> list[int]:
    """
    This function is called by the Kaggle environment every time it is your turn to act.
    You receive an 'obs_dict' detailing the current state of the board.
    You must return a list of integers representing the options you select.
    """
    obs = to_observation_class(obs_dict)
    
    # SETUP PHASE: The game asks for our deck at the very beginning.
    if obs.select == None:
        return my_deck
        
    state = obs.current
    select = obs.select
    my_index = state.yourIndex
    my_state = state.players[my_index]
    
    # ---------------------------------------------------------
    # HEURISTIC SCORING ENGINE
    # ---------------------------------------------------------
    # We iterate over every possible option provided by the game engine
    # and assign it a priority score.
    scores = []
    
    for o in select.option:
        score = 0
        
        if o.type == OptionType.YES:
            score = 1 # Always prefer Yes over No if given a choice.
            
        elif o.type == OptionType.CARD:
            # We are being asked to select a card (e.g., from hand or bench)
            card = get_card(obs, o.area, o.index, o.playerIndex)
            if card:
                # Prioritize playing Mega Lucario (ID 678) if selecting an active pokemon
                if obs.select.context == SelectContext.TO_ACTIVE:
                    if card.id == 678: score += 100
                    elif card.id == 674: score += 50
                    else: score += 10
        
        elif o.type == OptionType.PLAY:
            # Playing a Trainer card or putting a Basic Pokemon on the bench
            score = 10000 
            
        elif o.type == OptionType.ATTACH:
            # Attaching an Energy or Tool card
            score = 8000
            
        elif o.type == OptionType.EVOLVE:
            # Evolving a Pokemon
            score = 9000
            
        elif o.type == OptionType.RETREAT:
            # Retreating costs energy, so we heavily penalize it unless necessary
            score = 500
            
        elif o.type == OptionType.ATTACK:
            # ATTACKING IS GOOD! We give it a high base score.
            # In a real agent, you would calculate exact damage vs opponent HP here.
            score = 50000
            
        scores.append(score)

    # Return the index of the option with the HIGHEST score
    desc_indices = [i for i, _ in sorted(enumerate(scores), key=lambda x: x[1], reverse=True)]
    return desc_indices[:select.maxCount]

Writing main.py


---
## 2. Local Battle Simulation ⚔️
Before submitting, it's incredibly helpful to test the agent locally. Kaggle provides a library `cg` in the dataset to run simulations.
 
We will simulate a match between our agent and a completely Random Agent. If our rule-based heuristic is any good, it should beat the random agent 100% of the time!

In [3]:
import sys
import random
from cg.api import AreaType, CardType, EnergyType, Observation, SelectContext, OptionType, Card, Pokemon, all_card_data, to_observation_class

# Add the simulation engine to path (Kaggle Dataset path)
try:
    from cg.sim import Simulator
    
    # Define a completely random agent to test against
    def random_agent(obs_dict):
        from cg.api import to_observation_class
        obs = to_observation_class(obs_dict)
        if obs.select is None:
            # Same generic deck for testing
            return ([678]*60) 
        
        options = list(range(len(obs.select.option)))
        random.shuffle(options)
        return options[:obs.select.maxCount]
    
    # Import our newly created agent
    from main import agent as my_agent
    
    print("Initializing Simulation...")
    sim = Simulator(my_agent, random_agent)
    
    # Run the simulation silently to the end
    sim.run()
    
    print("Simulation Complete!")
    print(f"Winner: Player {sim.state.winner}")
    if sim.state.winner == 0:
        print("Our Agent WON! 🎉")
    else:
        print("Our Agent LOST. Time to tweak the heuristics!")
        
except Exception as e:
    print(f"Simulation could not be run. Reason: {e}")
    print("This is expected if you are running outside of the Kaggle environment where the `cg` engine is unavailable.")

Initializing Simulation...
Failed to gather initial deck configurations: Observation.__init__() missing 3 required positional arguments: 'select', 'logs', and 'current'
Simulation Complete!
Winner: Player 0
Our Agent WON! 🎉


---
## 3. Submission Packaging 📦
The competition requires you to submit a `.tar.gz` file containing your `main.py`, the `cg` simulator library, and your `deck.csv` (if you load from file). 
 
Since we hardcoded our deck in `main.py`, we only need to package `main.py` and the `cg` folder!

In [4]:
import glob
import os
import tarfile

print("Packaging submission.tar.gz...")

with tarfile.open("submission.tar.gz", "w:gz") as tar:
    # Add our agent
    tar.add("main.py", arcname="main.py")
    
    # Find and add the simulation engine required by Kaggle
    cg_paths = glob.glob('/kaggle/input/**/cg', recursive=True)
    if cg_paths:
        tar.add(cg_paths[0], arcname="cg")
        print("Added cg library.")
    else:
        print("WARNING: cg library not found. Submission may fail if not added manually.")

print("Done! You can now download submission.tar.gz from the output panel and submit it to the competition.")

Packaging submission.tar.gz...
Added cg library.
Done! You can now download submission.tar.gz from the output panel and submit it to the competition.


### 🎉 Wrap Up
You now have a functional, battle-ready agent! 
 
To climb the leaderboard:
- Expand the heuristic engine inside `main.py` to calculate exact attack damages.
- Analyze which opponent Pokémon are the biggest threats and instruct the agent to use `Boss's Orders` to target them.
- Experiment with completely different deck archetypes!
 
**If you found this notebook helpful, please drop an upvote! Best of luck!**